# 08 - In-domain ablation study

Runs the four configurations compared in the paper on the held-out in-domain
test split:

1. RoBERTa-LoRA only.
2. Classifier + k-NN probability interpolation.
3. Full RAG (classifier + FAISS evidence + LLM).
4. Full RAG with interpolation.
5. Full RAG with the light prompt (no retrieval evidence).

The LLM must be reachable through Ollama. Start it separately:

```bash
ollama serve
ollama pull llama3.1:8b-instruct-q8_0
```

## Setup

In [ ]:
import pandas as pd

from spam_detection.classifier import load_roberta_model, predict_roberta_proba
from spam_detection.config import (
    CLASSIFIER_TEST_EMBEDDINGS,
    CLASSIFIER_TEST_PATH,
    K_NEIGHBORS,
    RESULTS_DIR,
    TAU_HIGH,
    TAU_LOW,
    VECTOR_DB_INDEX,
    VECTOR_DB_PARQUET,
)
from spam_detection.evaluation import comparison_table, gray_zone_analysis, predictions_from_results, save_table
from spam_detection.pipeline import run_ablation_study
from spam_detection.prompts import create_hybrid_chain, create_light_chain
from spam_detection.retrieval import load_labeled_faiss_db, precompute_faiss_summaries
from spam_detection.utils import set_seed

set_seed(42)
RUN_LLM = True
OUTPUT_DIR = RESULTS_DIR / "runs" / "in_domain"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load data, classifier probabilities and retrieval neighbours

In [ ]:
df = pd.read_csv(CLASSIFIER_TEST_PATH)
y_true = df["label"].to_numpy()
X_emb = __import__("numpy").load(CLASSIFIER_TEST_EMBEDDINGS)

model, tokenizer = load_roberta_model()
clf_probas = predict_roberta_proba(df["text"].tolist(), model, tokenizer)
print("Reviews:", len(df), "| mean fake probability:", round(float(clf_probas.mean()), 4))

faiss_db = load_labeled_faiss_db(VECTOR_DB_PARQUET, VECTOR_DB_INDEX)
faiss_summaries = precompute_faiss_summaries(X_emb, faiss_db, k_neighbors=K_NEIGHBORS)

## Configuration 1 - classifier only

In [ ]:
clf_only = run_ablation_study(
    df=df,
    clf_probas=clf_probas,
    true_labels=y_true,
    use_llm=False,
    use_retrieval=False,
    use_interpolation=False,
    show_progress=False,
)
save_table(clf_only, OUTPUT_DIR / "clf_only.csv")

## Configuration 2 - k-NN interpolation

In [ ]:
interpolation = run_ablation_study(
    df=df,
    clf_probas=clf_probas,
    true_labels=y_true,
    faiss_summaries=faiss_summaries,
    use_llm=False,
    use_retrieval=False,
    use_interpolation=True,
    show_progress=False,
)
save_table(interpolation, OUTPUT_DIR / "interpolation.csv")

## Configurations 3-5 - full RAG

In [ ]:
if RUN_LLM:
    hybrid_chain = create_hybrid_chain()
    full_rag = run_ablation_study(
        df=df,
        clf_probas=clf_probas,
        true_labels=y_true,
        llm_chain=hybrid_chain,
        faiss_summaries=faiss_summaries,
        lower_thresh=TAU_LOW,
        upper_thresh=TAU_HIGH,
        use_llm=True,
        use_retrieval=True,
        use_interpolation=False,
    )
    save_table(full_rag, OUTPUT_DIR / "full_rag.csv")

    interpolated_rag = run_ablation_study(
        df=df,
        clf_probas=clf_probas,
        true_labels=y_true,
        llm_chain=hybrid_chain,
        faiss_summaries=faiss_summaries,
        lower_thresh=TAU_LOW,
        upper_thresh=TAU_HIGH,
        use_llm=True,
        use_retrieval=True,
        use_interpolation=True,
    )
    save_table(interpolated_rag, OUTPUT_DIR / "full_rag_interpolated.csv")

    light_chain = create_light_chain()
    light_rag = run_ablation_study(
        df=df,
        clf_probas=clf_probas,
        true_labels=y_true,
        llm_chain=light_chain,
        faiss_summaries=faiss_summaries,
        lower_thresh=TAU_LOW,
        upper_thresh=TAU_HIGH,
        use_llm=True,
        use_retrieval=True,
        use_interpolation=False,
    )
    save_table(light_rag, OUTPUT_DIR / "full_rag_light.csv")

## Metrics and uncertainty-window analysis

In [ ]:
predictions = {
    "RoBERTa-LoRA": predictions_from_results(clf_only),
    "Interpolation (k-NN + LoRA)": predictions_from_results(interpolation),
}
if RUN_LLM:
    predictions.update(
        {
            "Full RAG": predictions_from_results(full_rag),
            "Full RAG + interpolation": predictions_from_results(interpolated_rag),
            "Full RAG (light prompt)": predictions_from_results(light_rag),
        }
    )

table = comparison_table(y_true, predictions, dataset_name="In-Domain")
display(table)
save_table(table, RESULTS_DIR / "tables" / "in_domain_ablation.csv")

if RUN_LLM:
    print("Gray-zone metrics (classic RAG):")
    print(gray_zone_analysis(full_rag, TAU_LOW, TAU_HIGH))